In [7]:
%pip install kagglehub

import kagglehub

# Correct dataset: grassknoted/asl-alphabet
path = kagglehub.dataset_download("grassknoted/asl-alphabet")

print("Dataset Downloaded to:", path)

Note: you may need to restart the kernel to use updated packages.
Dataset Downloaded to: C:\Users\USER\.cache\kagglehub\datasets\grassknoted\asl-alphabet\versions\1


In [8]:
import os

DATASET_PATH = "/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train"
if os.path.exists(DATASET_PATH):
    classes = sorted(os.listdir(DATASET_PATH))
    print(f"Total Classes: {len(classes)}")
    print("Classes:", classes)

In [9]:
!pip install mediapipe

You should consider upgrading via the 'C:\Users\USER\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [13]:
import os
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Download MediaPipe Hand Landmarker model bundle
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

# Setup Task Landmarker options
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1)
detector = vision.HandLandmarker.create_from_options(options)

dataset_dir = kagglehub.dataset_download("grassknoted/asl-alphabet")
DATASET_PATH = os.path.join(dataset_dir, "asl_alphabet_train", "asl_alphabet_train")
# DATASET_PATH = "/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train"
classes = sorted(os.listdir(DATASET_PATH))

X_data = []
y_data = []
IMAGES_PER_CLASS = 500  # Fast processing සඳහා class එකකට images 500ක්

print("Extracting Landmarks...")

for class_name in classes:
    class_dir = os.path.join(DATASET_PATH, class_name)
    image_files = os.listdir(class_dir)[:IMAGES_PER_CLASS]
    print(f"Processing Class: {class_name}")

    for img_name in image_files:
        img_path = os.path.join(class_dir, img_name)

        # Load image via MediaPipe Image object
        mp_image = mp.Image.create_from_file(img_path)
        detection_result = detector.detect(mp_image)

        if detection_result.hand_landmarks:
            hand_landmarks = detection_result.hand_landmarks[0]

            # Wrist Relative Normalization (Point 0 relative)
            wrist_x = hand_landmarks[0].x
            wrist_y = hand_landmarks[0].y
            wrist_z = hand_landmarks[0].z

            landmarks = []
            for lm in hand_landmarks:
                landmarks.extend([
                    lm.x - wrist_x,
                    lm.y - wrist_y,
                    lm.z - wrist_z
                ])

            X_data.append(landmarks)
            y_data.append(class_name)

X_data = np.array(X_data)
y_data = np.array(y_data)

print(f"\nExtraction Completed Successfully!")
print(f"Total Extracted Samples: {X_data.shape[0]}")

'wget' is not recognized as an internal or external command,
operable program or batch file.


Extracting Landmarks...
Processing Class: A
Processing Class: B
Processing Class: C
Processing Class: D
Processing Class: E
Processing Class: F
Processing Class: G
Processing Class: H
Processing Class: I
Processing Class: J
Processing Class: K
Processing Class: L
Processing Class: M
Processing Class: N
Processing Class: O
Processing Class: P
Processing Class: Q
Processing Class: R
Processing Class: S
Processing Class: T
Processing Class: U
Processing Class: V
Processing Class: W
Processing Class: X
Processing Class: Y
Processing Class: Z
Processing Class: del
Processing Class: nothing
Processing Class: space

Extraction Completed Successfully!
Total Extracted Samples: 12503


In [15]:
%pip install pandas


Note: you may need to restart the kernel to use updated packages.Collecting pandas
  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl.metadata (19 kB)
Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl (11.3 MB)

   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ---------------------------------------- 0/3 [pytz]
   ----------

   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- -

In [16]:
import pandas as pd

# 1. made DataFrame
df = pd.DataFrame(X_data)
df['label'] = y_data


columns = []
for i in range(21):
    columns.extend([f'lm{i}_x', f'lm{i}_y', f'lm{i}_z'])
columns.append('label')

df.columns = columns

# 3. Updated CSV Save
df.to_csv("asl_hand_landmarks.csv", index=False)

print("Updated asl_hand_landmarks.csv file saved successfully with landmark column names!")

Updated asl_hand_landmarks.csv file saved successfully with landmark column names!


In [17]:
df.head()

,lm0_x,lm0_y,lm0_z,lm1_x,lm1_y,lm1_z,lm2_x,lm2_y,lm2_z,lm3_x,...,lm18_x,lm18_y,lm18_z,lm19_x,lm19_y,lm19_z,lm20_x,lm20_y,lm20_z,label
0,0.0,0.0,0.0,0.123345,-0.071737,-0.029440,0.192214,-0.196992,-0.040889,0.214579,...,-0.060982,-0.263164,-0.104181,-0.050525,-0.180352,-0.094723,-0.048616,-0.127567,-0.071827,A
1,0.0,0.0,0.0,0.114824,-0.062552,-0.034523,0.194717,-0.191315,-0.049414,0.219391,...,-0.053674,-0.251842,-0.116292,-0.045062,-0.168029,-0.107700,-0.046714,-0.118722,-0.088338,A
2,0.0,0.0,0.0,0.086921,-0.053995,-0.028928,0.137295,-0.162368,-0.036933,0.146925,...,-0.088018,-0.214530,-0.075172,-0.080280,-0.157623,-0.071258,-0.070828,-0.110785,-0.055775,A
3,0.0,0.0,0.0,0.156900,-0.091643,-0.039910,0.235989,-0.250556,-0.057184,0.231617,...,-0.054150,-0.302455,-0.127160,-0.029973,-0.214135,-0.123407,-0.024047,-0.156386,-0.106048,A
4,0.0,0.0,0.0,0.155789,-0.087835,-0.041631,0.236888,-0.247277,-0.058902,0.239514,...,-0.049126,-0.310105,-0.125537,-0.026262,-0.219725,-0.124559,-0.019785,-0.157306,-0.107887,A
